## Setup

In [30]:
# Install necessary libraries
!pip install pdfplumber pandas pymupdf tabulate

In [31]:
# Import necessary libraries
import fitz # PyMuPDF
import pdfplumber
import pandas as pd
from pathlib import Path
from tabulate import tabulate

### PDF Path Configuration

Specify the path to your PDF document. Make sure the file exists in your Colab environment. If your PDF is in Google Drive, you might need to mount your Drive first.

In [32]:
# Update PDF_PATH with the correct path to your uploaded PDF
# For example, if your file is 'sample_policy.pdf' in '/content/data/policies/'
PDF_PATH = "/content/pw_msme_pdf.pdf"

print(f"Updated PDF_PATH: {PDF_PATH}")

Updated PDF_PATH: /content/pw_msme_pdf.pdf


In [33]:
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Total Pages: {len(pdf.pages)}")

Total Pages: 35


## Extract Text Blocks from PDF

In [34]:
def extract_text_blocks(pdf_path):
    doc = fitz.open(pdf_path)

    pages_data = []

    for page_num in range(len(doc)):
        page = doc[page_num]

        blocks = page.get_text("blocks")

        page_text = []

        for block in blocks:
            x0, y0, x1, y1, text, *_ = block

            text = text.strip()

            if text:
                page_text.append(text)

        pages_data.append(
            {
                "page_number": page_num + 1,
                "text": "\n\n".join(page_text),
            }
        )

    return pages_data

In [35]:
pages = extract_text_blocks(PDF_PATH)

print("Pages:", len(pages))

print("\nFirst Page Preview:\n")
print(pages[0]["text"][:5000])

Pages: 35

First Page Preview:

ICICI Lombard General Insurance Company Limited 
ICICI Lombard MSME Suraksha Kavach (Complete Fire Insurance) 
IRDA Reg No. 115                                                     CIN: L67200MH2000PLC129408                                                                   UIN – IRDAN115RPPR0010V01202425 
Mailing Address:                                                         Registered Office Address:                                                                             Toll Free no    : 1800 2666 
601 & 602, 6th Floor, Interface 16,                             ICICI Lombard House, 414, Veer Savarkar,                                                    Alternate No: 86552 22666 
New Linking Road, Malad (West)                                            Marg, Near Siddhi Vinayak Temple, Prabhadevi,                                                               E-mail: customersupport@icicilombard.com  
Mumbai – 400064                                  

## Extract Tables from PDF

In [36]:
def extract_tables(pdf_path):

    extracted_tables = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            tables = page.extract_tables()

            if not tables:
                continue

            for table_idx, table in enumerate(tables):

                try:

                    df = pd.DataFrame(table)

                    extracted_tables.append(
                        {
                            "page_number": page_num,
                            "table_index": table_idx,
                            "dataframe": df,
                            "column_count": len(df.columns)
                        }
                    )

                except Exception:
                    pass

    return extracted_tables

In [37]:
def looks_like_header(row):

    row = [str(x).strip() if x else "" for x in row]

    keywords = [
        "word",
        "meaning",
        "description",
        "occupancy",
        "deductible",
        "minimum",
        "maximum",
        "coverage"
    ]

    row_text = " ".join(row).lower()

    return any(k in row_text for k in keywords)

In [38]:
def merge_continued_tables(tables):

    merged = []

    current = None

    for table in tables:

        df = table["dataframe"]

        first_row = df.iloc[0].tolist()

        has_header = looks_like_header(first_row)

        if current is None:

            current = table.copy()

            current["pages"] = [table["page_number"]]

            continue

        same_columns = (
            table["column_count"]
            == current["column_count"]
        )

        next_page = (
            table["page_number"]
            == current["pages"][-1] + 1
        )

        continuation = (
            same_columns
            and next_page
            and not has_header
        )

        if continuation:

            current["dataframe"] = pd.concat(
                [
                    current["dataframe"],
                    df
                ],
                ignore_index=True
            )

            current["pages"].append(
                table["page_number"]
            )

        else:

            merged.append(current)

            current = table.copy()

            current["pages"] = [table["page_number"]]

    if current:

        merged.append(current)

    return merged

In [39]:
tables = extract_tables(PDF_PATH)

merged_tables = merge_continued_tables(
    tables
)

print(
    "Original Tables:",
    len(tables)
)

print(
    "Merged Tables:",
    len(merged_tables)
)

Original Tables: 55
Merged Tables: 29


In [40]:
for table in merged_tables:

    print("=" * 100)

    print(
        "Pages:",
        table["pages"]
    )

    display(
        table["dataframe"].head()
    )

Pages: [1]


,0,1
0,Word/s,Specific meaning
1,Agreed\nValue,An amount agreed between\nYou and Us at the po...
2,Bank,A bank or any financial\ninstitution


Pages: [1, 2]


,0,1
0,,
1,a.\nBuilding\nb.\ni.,Any building or structure in\nYour Premises wh...


Pages: [2]


...

,0,1
0,,


Pages: [35]


,0
0,Tel.: 0120-2514252 / 2514253\nEmail: bimalokpa...
1,PATNA\nInsurance Ombudsman\nOffice of the Insu...
2,Pune\nInsurance Ombudsman\nOffice of the Insur...
3,THANE\nInsurance Ombudsman Office of the\nInsu...


Pages: [35]


,0,1
0,,


## Reconstruct Document

In [41]:
def table_to_text(df):

    rows = []

    headers = [
        str(x).strip()
        for x in df.iloc[0]
    ]

    for _, row in df.iloc[1:].iterrows():

        pairs = []

        for h, value in zip(
            headers,
            row.tolist()
        ):

            pairs.append(
                f"{h}: {value}"
            )

        rows.append(
            " | ".join(pairs)
        )

    return "\n".join(rows)

In [42]:
def build_reconstructed_document(
    pages,
    merged_tables
):

    document = []

    for page in pages:

        page_num = page["page_number"]

        content = []

        content.append(
            f"\n===== PAGE {page_num} =====\n"
        )

        content.append(page["text"])

        page_tables = [
            t for t in merged_tables
            if page_num in t["pages"]
        ]

        for table in page_tables:

            content.append(
                "\n\n[TABLE]\n"
            )

            content.append(
                table_to_text(
                    table["dataframe"]
                )
            )

        document.append(
            "\n".join(content)
        )

    return "\n\n".join(document)

In [43]:
reconstructed_document = build_reconstructed_document(
    pages,
    merged_tables
)

In [44]:
print(reconstructed_document[:5000])


===== PAGE 1 =====

ICICI Lombard General Insurance Company Limited 
ICICI Lombard MSME Suraksha Kavach (Complete Fire Insurance) 
IRDA Reg No. 115                                                     CIN: L67200MH2000PLC129408                                                                   UIN – IRDAN115RPPR0010V01202425 
Mailing Address:                                                         Registered Office Address:                                                                             Toll Free no    : 1800 2666 
601 & 602, 6th Floor, Interface 16,                             ICICI Lombard House, 414, Veer Savarkar,                                                    Alternate No: 86552 22666 
New Linking Road, Malad (West)                                            Marg, Near Siddhi Vinayak Temple, Prabhadevi,                                                               E-mail: customersupport@icicilombard.com  
Mumbai – 400064                                             

In [45]:
with open(
    "reconstructed_document.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        reconstructed_document
    )

print("Saved Successfully")

Saved Successfully
